# Part A — Phase 1: Load & Merge

In [1]:
import pandas as pd
import numpy as np

h1 = pd.read_csv("../data/raw/H1.csv")
h2 = pd.read_csv("../data/raw/H2.csv")
print("H1 shape:", h1.shape)
print("H2 shape:", h2.shape) #to verify that they have loaded correctly

H1 shape: (40060, 31)
H2 shape: (79330, 31)


In [3]:
h1["Hotel"] = "Resort Hotel"
h2["Hotel"] = "City Hotel"
# to check which ds came from which hotel
print(h1["Hotel"].unique())
print(h2["Hotel"].unique())
# we added one more column so shape should change
print("H1 shape:", h1.shape)
print("H2 shape:", h2.shape)

['Resort Hotel']
['City Hotel']
H1 shape: (40060, 32)
H2 shape: (79330, 32)


In [4]:
df= pd.concat([h1,h2],ignore_index=True)
#ignore_index --Don't preserve the old H1/H2 row indices. Create a fresh 0 → 119389 index.
print("Combined shape:", df.shape)

Combined shape: (119390, 32)


In [ ]:
# assert only produces something if the condition is false.
# this section is only for verifying
expected_rows = 40060 + 79330
assert len(df) == expected_rows
print("Row count verified:", len(df))

print("Final shape:", df.shape)
print("\nHotel distribution:")
print(df["Hotel"].value_counts())

Row count verified: 119390
Final shape: (119390, 32)

Hotel distribution:
Hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64


# Part B — Phase 2: Inspect Before Cleaning

In [ ]:
#INSPECT THE DATA BEFORE CLEANING

print("general structure:")
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\ndata types:")
print(df.dtypes)

print("\nmissing values:")
print(df.isna().sum())

print("\n hidden 'missing' values:")
print("NULL values in Agent:", (df["Agent"] == "NULL").sum())
print("NULL values in Company:", (df["Company"] == "NULL").sum())

print("\n categorical columns:")

categorical_columns = df.select_dtypes(include="object").columns
for col in categorical_columns:
    print("\n", col)
    print("Number of unique values:", df[col].nunique())
    print("Sample values:", df[col].unique()[:10])


print("\n whitespace check:")

for col in categorical_columns:
    if df[col].astype(str).str.strip().ne(df[col].astype(str)).any():
        print(col, "contains extra whitespace")

print("Number of exact duplicate rows:", df.duplicated().sum())


print("\n numerical variables:")

numeric_columns = df.select_dtypes(include=np.number).columns
print("Numerical columns:")
print(numeric_columns.tolist())

print("\nBasic numerical summary:")
display(df[numeric_columns].describe().T)


print("\n ADR check:")

print("Negative ADR values:", (df["ADR"] < 0).sum())
print("Zero ADR values:", (df["ADR"] == 0).sum())
print("Minimum ADR:", df["ADR"].min())
print("Maximum ADR:", df["ADR"].max())


print("\n ADR extreme values:")

print("Smallest ADR values:")
print(df["ADR"].nsmallest(10))

print("\nLargest ADR values:")
print(df["ADR"].nlargest(10))


print("\n checking numerical ranges:")

check_columns = [
    "LeadTime",
    "StaysInWeekendNights",
    "StaysInWeekNights",
    "Adults",
    "Children",
    "Babies",
    "PreviousCancellations",
    "PreviousBookingsNotCanceled",
    "BookingChanges",
    "DaysInWaitingList",
    "RequiredCarParkingSpaces",
    "TotalOfSpecialRequests"
]

for col in check_columns:
    print(f"{col}: min = {df[col].min()}, max = {df[col].max()}")

general structure:
Shape of dataset: (119390, 32)

First 5 rows:


,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate,Hotel
0,0,342,2015,July,27,1,0,0,2,0.0,...,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01,Resort Hotel
1,0,737,2015,July,27,1,0,0,2,0.0,...,NULL,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-01,Resort Hotel
2,0,7,2015,July,27,1,0,1,1,0.0,...,NULL,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02,Resort Hotel
3,0,13,2015,July,27,1,0,1,1,0.0,...,304,NULL,0,Transient,75.0,0,0,Check-Out,2015-07-02,Resort Hotel
4,0,14,2015,July,27,1,0,2,2,0.0,...,240,NULL,0,Transient,98.0,0,1,Check-Out,2015-07-03,Resort Hotel



data types:
IsCanceled                       int64
LeadTime                         int64
ArrivalDateYear                  int64
ArrivalDateMonth                object
ArrivalDateWeekNumber            int64
ArrivalDateDayOfMonth            int64
StaysInWeekendNights             int64
StaysInWeekNights                int64
Adults                           int64
Children                       float64
Babies                           int64
Meal                            object
Country                         object
MarketSegment                   object
DistributionChannel             object
IsRepeatedGuest                  int64
PreviousCancellations            int64
PreviousBookingsNotCanceled      int64
ReservedRoomType                object
AssignedRoomType                object
BookingChanges                   int64
DepositType                     object
Agent                           object
Company                         object
DaysInWaitingList                int64
CustomerType

,count,mean,std,min,25%,50%,75%,max
IsCanceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
LeadTime,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
ArrivalDateYear,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
ArrivalDateWeekNumber,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
ArrivalDateDayOfMonth,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
StaysInWeekendNights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
StaysInWeekNights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
Adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
Children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
Babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0



 ADR check:
Negative ADR values: 1
Zero ADR values: 1959
Minimum ADR: -6.38
Maximum ADR: 5400.0

 ADR extreme values:
Smallest ADR values:
14969   -6.38
0        0.00
1        0.00
125      0.00
167      0.00
168      0.00
196      0.00
197      0.00
421      0.00
428      0.00
Name: ADR, dtype: float64

Largest ADR values:
48515     5400.00
111403     510.00
15083      508.00
103912     451.50
13142      450.00
13391      437.00
39155      426.25
39568      402.00
39118      397.38
13323      392.00
Name: ADR, dtype: float64

 checking numerical ranges:
LeadTime: min = 0, max = 737
StaysInWeekendNights: min = 0, max = 19
StaysInWeekNights: min = 0, max = 50
Adults: min = 0, max = 55
Children: min = 0.0, max = 10.0
Babies: min = 0, max = 10
PreviousCancellations: min = 0, max = 26
PreviousBookingsNotCanceled: min = 0, max = 72
BookingChanges: min = 0, max = 21
DaysInWaitingList: min = 0, max = 391
RequiredCarParkingSpaces: min = 0, max = 8
TotalOfSpecialRequests: min = 0, max = 5


ADR = Average Daily Rate.
In this hotel dataset, ADR represents the average daily price/rate associated with a booking.

Simple example
Suppose a booking has:
3 nights
Total room price = ₹30,000
Then roughly:
$$ ADR = \frac{30,000}{3} = ₹10,000 $$

So ADR basically tells us how much the booking costs per day/night on average.
An extreme ADR is an ADR value that is unusually far from most of the other bookings.

# Part C — Cleaning

In [12]:
# Create a copy for cleaning
clean_df = df.copy()

print("Original shape:", clean_df.shape)
print("CLEANING WHITESPACE:")# checked above, cleaning now
object_columns = clean_df.select_dtypes(include="object").columns

for col in object_columns:
    clean_df[col] = clean_df[col].str.strip()

print("Whitespace cleaning completed.")

print("\n========== NULL PLACEHOLDERS ==========")

for col in ["Agent", "Company"]:
    null_count = clean_df[col].astype(str).str.strip().eq("NULL").sum()
    print(col, "NULL values:", null_count)
clean_df["Agent"] = clean_df["Agent"].replace("NULL", np.nan)
clean_df["Company"] = clean_df["Company"].replace("NULL", np.nan)

clean_df["Agent"] = clean_df["Agent"].fillna("Not Applicable")
clean_df["Company"] = clean_df["Company"].fillna("Not Applicable")


Original shape: (119390, 32)
CLEANING WHITESPACE:
Whitespace cleaning completed.

========== NULL PLACEHOLDERS ==========
Agent NULL values: 16340
Company NULL values: 112593


why do we do whitespace cleaning?Values such as:

"Online TA"

" Online TA"

"Online TA "

should represent the same category.

In [17]:
print("\n========== CHILDREN ==========")

print("Missing Children before cleaning:",
      clean_df["Children"].isna().sum())

clean_df["Children"] = clean_df["Children"].fillna(0)

print("Missing Children after cleaning:",
      clean_df["Children"].isna().sum())


print("\n========== C4: COUNTRY ==========")

print("Missing Country:",
      clean_df["Country"].isna().sum())
clean_df["Country"] = clean_df["Country"].fillna("Unknown")

print("Missing Country after cleaning:",
      clean_df["Country"].isna().sum())

print("\n========== C5: DUPLICATES ==========")

duplicate_count = clean_df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

before_duplicates = len(clean_df)
clean_df = clean_df.drop_duplicates()
after_duplicates = len(clean_df)
print("Rows before removing duplicates:", before_duplicates)
print("Rows after removing duplicates:", after_duplicates)
print("Rows removed:", before_duplicates - after_duplicates)


========== CHILDREN ==========
Missing Children before cleaning: 0
Missing Children after cleaning: 0

========== C4: COUNTRY ==========
Missing Country: 0
Missing Country after cleaning: 0

========== C5: DUPLICATES ==========
Duplicate rows: 31994
Rows before removing duplicates: 119390
Rows after removing duplicates: 87396
Rows removed: 31994


In [18]:
print("\n========== C6: ADULTS ==========")

print("Bookings with 0 adults:",
      (clean_df["Adults"] == 0).sum())

print("Bookings with more than 10 adults:",
      (clean_df["Adults"] > 10).sum())

print("\nBookings with 0 adults:")
print(
    clean_df[clean_df["Adults"] == 0]
    [["Hotel", "Adults", "Children", "Babies", "ADR", "IsCanceled"]]
    .head(10)
)

print("\nBookings with more than 10 adults:")
print(
    clean_df[clean_df["Adults"] > 10]
    [["Hotel", "Adults", "Children", "Babies", "ADR", "IsCanceled"]]
    .sort_values("Adults", ascending=False)
    .head(10)
)


========== C6: ADULTS ==========
Bookings with 0 adults: 385
Bookings with more than 10 adults: 12

Bookings with 0 adults:
              Hotel  Adults  Children  Babies   ADR  IsCanceled
2224   Resort Hotel       0       0.0       0   0.0           0
2409   Resort Hotel       0       0.0       0   0.0           0
3181   Resort Hotel       0       0.0       0   0.0           0
3684   Resort Hotel       0       0.0       0   0.0           0
3708   Resort Hotel       0       0.0       0   0.0           0
4127   Resort Hotel       0       0.0       0   0.0           1
9376   Resort Hotel       0       0.0       0   0.0           1
31765  Resort Hotel       0       0.0       0  28.0           0
32029  Resort Hotel       0       0.0       0   0.0           0
32827  Resort Hotel       0       0.0       0   0.0           0

Bookings with more than 10 adults:
             Hotel  Adults  Children  Babies  ADR  IsCanceled
2173  Resort Hotel      55       0.0       0  0.0           1
1643  Resor

In [21]:
print("\n========== C7: ADR ==========")

negative_adr = clean_df[clean_df["ADR"] < 0]

print("Negative ADR rows:", len(negative_adr))

print(
    negative_adr[
        ["Hotel", "IsCanceled", "Adults", "Children",
         "StaysInWeekendNights", "StaysInWeekNights", "ADR"]
    ]
)

clean_df.loc[clean_df["ADR"] < 0, "ADR"] = np.nan
clean_df["ADR"] = clean_df.groupby("Hotel")["ADR"].transform(
    lambda x: x.fillna(x.median())
)

clean_df["ADR"] = clean_df.groupby("Hotel")["ADR"].transform(
    lambda x: x.fillna(x.median())
)

print("Negative ADR after cleaning:",
      (clean_df["ADR"] < 0).sum())

print("Missing ADR after cleaning:",
      clean_df["ADR"].isna().sum())

print("Zero ADR values:", (clean_df["ADR"] == 0).sum())


========== C7: ADR ==========
Negative ADR rows: 0
Empty DataFrame
Columns: [Hotel, IsCanceled, Adults, Children, StaysInWeekendNights, StaysInWeekNights, ADR]
Index: []
Negative ADR after cleaning: 0
Missing ADR after cleaning: 0
Zero ADR values: 1778


In [23]:
print("ADR values above 1000:",
      (clean_df["ADR"] > 1000).sum())

print(
    clean_df[clean_df["ADR"] > 1000]
    [["Hotel", "ADR", "Adults", "Children",
      "StaysInWeekendNights", "StaysInWeekNights"]]
)

print("\n========== C10: FINAL MISSING VALUE CHECK ==========")

missing_values = clean_df.isna().sum()
print(missing_values[missing_values > 0])



print("\n========== CLEANED DATASET ==========")

print("Cleaned shape:", clean_df.shape)

ADR values above 1000: 1
            Hotel     ADR  Adults  Children  StaysInWeekendNights  \
48515  City Hotel  5400.0       2       0.0                     0   

       StaysInWeekNights  
48515                  1  

========== C10: FINAL MISSING VALUE CHECK ==========
Series([], dtype: int64)

========== CLEANED DATASET ==========
Cleaned shape: (87396, 32)


# Part D — Feature Engineering

In [26]:
#creating a few new useful columns from the existing data
print("========== D1: ARRIVAL DATE ==========")
# currenty the below 3 columns are stored separately, we want to combine it under 1 column ArrivalDate
clean_df["ArrivalDate"] = pd.to_datetime(
    clean_df["ArrivalDateYear"].astype(str) + "-" +
    clean_df["ArrivalDateMonth"] + "-" +
    clean_df["ArrivalDateDayOfMonth"].astype(str)
)

print(clean_df["ArrivalDate"].head())

print("\n========== D2: TOTAL NIGHTS ==========")
# again combining the below 2 columns to make 1 column TotalNights
clean_df["TotalNights"] = (
    clean_df["StaysInWeekendNights"] +
    clean_df["StaysInWeekNights"]
)

print(clean_df["TotalNights"].head())


print("\n========== D3: TOTAL GUESTS ==========")
# combining all different type of guests into 1 column TotalGuests
clean_df["TotalGuests"] = (
    clean_df["Adults"] +
    clean_df["Children"] +
    clean_df["Babies"]
)

print(clean_df["TotalGuests"].head())

========== D1: ARRIVAL DATE ==========
0   2015-07-01
1   2015-07-01
2   2015-07-01
3   2015-07-01
4   2015-07-01
Name: ArrivalDate, dtype: datetime64[ns]

========== D2: TOTAL NIGHTS ==========
0    0
1    0
2    1
3    1
4    2
Name: TotalNights, dtype: int64

========== D3: TOTAL GUESTS ==========
0    2.0
1    2.0
2    1.0
3    1.0
4    2.0
Name: TotalGuests, dtype: float64


NOTE
LeadTime tells us how many days before arrival the booking was made.
But looking at every individual value from 0 to 737 isn't always convenient.
So we divide it into meaningful groups.
|   Lead Time | Band    |
| ----------: | ------- |
|    0–7 days | `0-7d`  |
|   8–30 days | `1-4wk` |
|  31–90 days | `1-3mo` |
| 91–180 days | `3-6mo` |
|   181+ days | `6mo+`  |


In [ ]:
print("\n========== D4: LEAD TIME BAND ==========")

clean_df["LeadTimeBand"] = pd.cut(
    clean_df["LeadTime"],
    bins=[-1, 7, 30, 90, 180, 1000],
    labels=["0-7d", "1-4wk", "1-3mo", "3-6mo", "6mo+"]
)

print(clean_df["LeadTimeBand"].value_counts().sort_index())

#Why -1 as the first boundary?
#Because LeadTime can start at 0.


========== D4: LEAD TIME BAND ==========
LeadTimeBand
0-7d     18304
1-4wk    16340
1-3mo    22744
3-6mo    18243
6mo+     11765
Name: count, dtype: int64


In [28]:
print("\n========== D5: FEATURE CHECK ==========")

print(clean_df[
    [
        "ArrivalDate",
        "TotalNights",
        "TotalGuests",
        "LeadTime",
        "LeadTimeBand"
    ]
].head(10))
print("\n========== D6: FEATURE MISSING VALUES ==========")

print(clean_df[
    ["ArrivalDate", "TotalNights", "TotalGuests", "LeadTimeBand"]
].isna().sum())

print("\n========== FINAL PART D SHAPE ==========")
print(clean_df.shape)


========== D5: FEATURE CHECK ==========
   ArrivalDate  TotalNights  TotalGuests  LeadTime LeadTimeBand
0   2015-07-01            0          2.0       342         6mo+
1   2015-07-01            0          2.0       737         6mo+
2   2015-07-01            1          1.0         7         0-7d
3   2015-07-01            1          1.0        13        1-4wk
4   2015-07-01            2          2.0        14        1-4wk
6   2015-07-01            2          2.0         0         0-7d
7   2015-07-01            2          2.0         9        1-4wk
8   2015-07-01            3          2.0        85        1-3mo
9   2015-07-01            3          2.0        75        1-3mo
10  2015-07-01            4          2.0        23        1-4wk

========== D6: FEATURE MISSING VALUES ==========
ArrivalDate     0
TotalNights     0
TotalGuests     0
LeadTimeBand    0
dtype: int64

========== FINAL PART D SHAPE ==========
(87396, 36)


# Part E — Descriptive Statistics

Columns chosen for descriptive stats:

LeadTime → booking timing

TotalNights → length of stay

TotalGuests → party size

ADR → daily room rate

BookingChanges → changes made to booking

DaysInWaitingList → waiting period

TotalOfSpecialRequests → number of special requests

In [32]:
print("========== E1: NUMERICAL VARIABLES ==========")

descriptive_columns = [
    "LeadTime",
    "TotalNights",
    "TotalGuests",
    "ADR",
    "BookingChanges",
    "DaysInWaitingList",
    "TotalOfSpecialRequests"
]

print("Variables selected for descriptive statistics:")
print(descriptive_columns)

print("\n========== E2: BASIC DESCRIPTIVE STATISTICS ==========")

print(clean_df[descriptive_columns].describe().T)

========== E1: NUMERICAL VARIABLES ==========
Variables selected for descriptive statistics:
['LeadTime', 'TotalNights', 'TotalGuests', 'ADR', 'BookingChanges', 'DaysInWaitingList', 'TotalOfSpecialRequests']

========== E2: BASIC DESCRIPTIVE STATISTICS ==========
                          count        mean        std  min   25%   50%  \
LeadTime                87396.0   79.891368  86.052325  0.0  11.0  49.0   
TotalNights             87396.0    3.630658   2.763449  0.0   2.0   3.0   
TotalGuests             87396.0    2.025253   0.794319  0.0   2.0   2.0   
ADR                     87396.0  106.338229  55.012707  0.0  72.0  98.1   
BookingChanges          87396.0    0.271603   0.727245  0.0   0.0   0.0   
DaysInWaitingList       87396.0    0.749565  10.015731  0.0   0.0   0.0   
TotalOfSpecialRequests  87396.0    0.698567   0.831946  0.0   0.0   0.0   

                          75%     max  
LeadTime                125.0   737.0  
TotalNights               5.0    69.0  
TotalGuests    

FIVE NUMBER SUMMARY

In [ ]:
print("\n========== E3: FIVE-NUMBER SUMMARY ==========")

five_number_summary = clean_df[descriptive_columns].describe().T[
    ["min", "25%", "50%", "75%", "max"]
]

print(five_number_summary)

print("\n========== E4: VARIANCE AND STANDARD DEVIATION ==========")

variance = clean_df[descriptive_columns].var()
std_dev = clean_df[descriptive_columns].std()

variance_std = pd.DataFrame({
    "Variance": variance,
    "Standard Deviation": std_dev
})

print(variance_std)

print("\n========== E5: SKEWNESS ==========")

skewness = clean_df[descriptive_columns].skew()

print(skewness)




========== E3: FIVE-NUMBER SUMMARY ==========
                        min   25%   50%    75%     max
LeadTime                0.0  11.0  49.0  125.0   737.0
TotalNights             0.0   2.0   3.0    5.0    69.0
TotalGuests             0.0   2.0   2.0    2.0    55.0
ADR                     0.0  72.0  98.1  134.0  5400.0
BookingChanges          0.0   0.0   0.0    0.0    21.0
DaysInWaitingList       0.0   0.0   0.0    0.0   391.0
TotalOfSpecialRequests  0.0   0.0   0.0    1.0     5.0

========== E4: VARIANCE AND STANDARD DEVIATION ==========
                           Variance  Standard Deviation
LeadTime                7405.002696           86.052325
TotalNights                7.636650            2.763449
TotalGuests                0.630943            0.794319
ADR                     3026.397921           55.012707
BookingChanges             0.528886            0.727245
DaysInWaitingList        100.314866           10.015731
TotalOfSpecialRequests     0.692134            0.831946

=====

| Skewness | General interpretation  |
| -------: | ----------------------- |
| Around 0 | Approximately symmetric |
| Positive | Right-skewed            |
| Negative | Left-skewed             |

Kurtosis-Kurtosis describes the tail behaviour / concentration of a distribution.
A simple interpretation:


Around 0 → similar tail behaviour to a normal distribution
Positive → heavier tails / more extreme observations
Negative → lighter tails

In [35]:
print("\n========== E6: KURTOSIS ==========")

kurtosis = clean_df[descriptive_columns].kurt()

print(kurtosis)



========== E6: KURTOSIS ==========
LeadTime                    2.126343
TotalNights                26.016498
TotalGuests               514.938864
ADR                       981.709025
BookingChanges             67.615632
DaysInWaitingList         480.609113
TotalOfSpecialRequests      0.819870
dtype: float64


Pearson correlation measures the strength and direction of a linear relationship between two numerical variables.

It ranges from:

$$ -1 \leq r \leq 1 $$
| Correlation | Meaning                              |
| ----------: | ------------------------------------ |
|        `+1` | Perfect positive linear relationship |
|         `0` | No linear relationship               |
|        `-1` | Perfect negative linear relationship |

#Correlation does not imply causation.

#Covariance indicates whether two variables tend to increase or decrease together.


In [39]:
print("\n========== E8: CORRELATION MATRIX ==========")

correlation_matrix = clean_df[descriptive_columns].corr()

print(correlation_matrix.round(3))



========== E8: CORRELATION MATRIX ==========
                        LeadTime  TotalNights  TotalGuests    ADR  \
LeadTime                   1.000        0.318        0.127  0.024   
TotalNights                0.318        1.000        0.104  0.054   
TotalGuests                0.127        0.104        1.000  0.387   
ADR                        0.024        0.054        0.387  1.000   
BookingChanges             0.077        0.082       -0.008  0.003   
DaysInWaitingList          0.132       -0.010       -0.025 -0.033   
TotalOfSpecialRequests     0.034        0.040        0.128  0.138   

                        BookingChanges  DaysInWaitingList  \
LeadTime                         0.077              0.132   
TotalNights                      0.082             -0.010   
TotalGuests                     -0.008             -0.025   
ADR                              0.003             -0.033   
BookingChanges                   1.000              0.024   
DaysInWaitingList                0.

In [40]:
print("\n========== E9: COVARIANCE MATRIX ==========")

covariance_matrix = clean_df[descriptive_columns].cov()

print(covariance_matrix.round(3))


========== E9: COVARIANCE MATRIX ==========
                        LeadTime  TotalNights  TotalGuests       ADR  \
LeadTime                7405.003       75.685        8.663   111.668   
TotalNights               75.685        7.637        0.227     8.240   
TotalGuests                8.663        0.227        0.631    16.912   
ADR                      111.668        8.240       16.912  3026.398   
BookingChanges             4.820        0.165       -0.005     0.104   
DaysInWaitingList        113.899       -0.288       -0.200   -18.185   
TotalOfSpecialRequests     2.451        0.092        0.085     6.308   

                        BookingChanges  DaysInWaitingList  \
LeadTime                         4.820            113.899   
TotalNights                      0.165             -0.288   
TotalGuests                     -0.005             -0.200   
ADR                              0.104            -18.185   
BookingChanges                   0.529              0.176   
DaysInWaitin

Identify the strongest correlations

In [41]:
print("\n========== E10: CORRELATION PAIRS ==========")

corr_pairs = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

corr_pairs = corr_pairs.stack().sort_values(
    key=abs,
    ascending=False
)

print(corr_pairs)


========== E10: CORRELATION PAIRS ==========
TotalGuests        ADR                       0.387018
LeadTime           TotalNights               0.318272
ADR                TotalOfSpecialRequests    0.137819
LeadTime           DaysInWaitingList         0.132152
TotalGuests        TotalOfSpecialRequests    0.128101
LeadTime           TotalGuests               0.126736
TotalNights        TotalGuests               0.103573
                   BookingChanges            0.081987
LeadTime           BookingChanges            0.077028
TotalNights        ADR                       0.054203
DaysInWaitingList  TotalOfSpecialRequests   -0.049545
TotalNights        TotalOfSpecialRequests    0.040204
LeadTime           TotalOfSpecialRequests    0.034240
ADR                DaysInWaitingList        -0.033003
TotalGuests        DaysInWaitingList        -0.025111
BookingChanges     DaysInWaitingList         0.024204
LeadTime           ADR                       0.023589
BookingChanges     TotalOfSpecialReq

In [ ]:
print("\n========== E11: MEAN VS MEDIAN ==========")

mean_median = pd.DataFrame({
    "Mean": clean_df[descriptive_columns].mean(),
    "Median": clean_df[descriptive_columns].median()
})

mean_median["Difference"] = (
    mean_median["Mean"] - mean_median["Median"]
)

print(mean_median)
#large difference between mean and median 
#can indicate that the variable is affected by skewed observations or extreme values.


========== E11: MEAN VS MEDIAN ==========
                              Mean  Median  Difference
LeadTime                 79.891368    49.0   30.891368
TotalNights               3.630658     3.0    0.630658
TotalGuests               2.025253     2.0    0.025253
ADR                     106.338229    98.1    8.238229
BookingChanges            0.271603     0.0    0.271603
DaysInWaitingList         0.749565     0.0    0.749565
TotalOfSpecialRequests    0.698567     0.0    0.698567


In [43]:
print("\n========== FINAL DESCRIPTIVE STATISTICS ==========")

final_stats = pd.DataFrame({
    "Min": clean_df[descriptive_columns].min(),
    "Q1": clean_df[descriptive_columns].quantile(0.25),
    "Median": clean_df[descriptive_columns].median(),
    "Q3": clean_df[descriptive_columns].quantile(0.75),
    "Max": clean_df[descriptive_columns].max(),
    "Mean": clean_df[descriptive_columns].mean(),
    "Variance": clean_df[descriptive_columns].var(),
    "Std Dev": clean_df[descriptive_columns].std(),
    "Skewness": clean_df[descriptive_columns].skew(),
    "Kurtosis": clean_df[descriptive_columns].kurt()
})

print(final_stats.round(3))


========== FINAL DESCRIPTIVE STATISTICS ==========
                        Min    Q1  Median     Q3     Max     Mean  Variance  \
LeadTime                0.0  11.0    49.0  125.0   737.0   79.891  7405.003   
TotalNights             0.0   2.0     3.0    5.0    69.0    3.631     7.637   
TotalGuests             0.0   2.0     2.0    2.0    55.0    2.025     0.631   
ADR                     0.0  72.0    98.1  134.0  5400.0  106.338  3026.398   
BookingChanges          0.0   0.0     0.0    0.0    21.0    0.272     0.529   
DaysInWaitingList       0.0   0.0     0.0    0.0   391.0    0.750   100.315   
TotalOfSpecialRequests  0.0   0.0     0.0    1.0     5.0    0.699     0.692   

                        Std Dev  Skewness  Kurtosis  
LeadTime                 86.052     1.432     2.126  
TotalNights               2.763     3.073    26.016  
TotalGuests               0.794    10.333   514.939  
ADR                      55.013    10.922   981.709  
BookingChanges            0.727     5.546    

In [44]:
print("========== FINAL DATA CHECK ==========")

print("Final shape:", clean_df.shape)
print("Duplicate rows:", clean_df.duplicated().sum())
print("Total missing values:", clean_df.isna().sum().sum())

print("\nNew features:")
print([
    "ArrivalDate",
    "TotalNights",
    "TotalGuests",
    "LeadTimeBand"
])

========== FINAL DATA CHECK ==========
Final shape: (87396, 36)
Duplicate rows: 0
Total missing values: 0

New features:
['ArrivalDate', 'TotalNights', 'TotalGuests', 'LeadTimeBand']


In [45]:
clean_df.to_csv("../data/processed/hotel_bookings_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
